# 2023년 제7차 근로환경조사(KWCS) 분석

## 단계: 01. 데이터 전처리 (Data Preprocessing)
- 목표: 5만 명 규모의 설문 원시자료를 분석 가능한 형태로 정제
- 데이터: 산업안전보건연구원 제7차 근로환경조사(2023), 50,195명 × 439개 변수
- 이전 프로젝트와의 차이: 지역 단위 집계 데이터 → 개인 단위 표본조사 마이크로데이터

### 1.1 원시자료 로드
- 확장자는 .csv이나 실제 구분자는 탭(\t)이며, 각 행 전체가 큰따옴표로 감싸져 있고 줄 끝에 쉼표 3개가 붙어 있음
- `quoting=csv.QUOTE_NONE`으로 따옴표를 무시하고 읽은 뒤, 첫/끝 컬럼의 잔여 문자를 제거
- `dtype=str`로 읽는 이유: 결측코드가 섞인 상태에서 숫자로 읽으면 컬럼별 타입 추론이 어긋남
- 공백 문자(' ')는 설문 미해당(skip pattern)을 의미하므로 결측(NA)으로 변환
- 원본 파일은 엑셀로 열지 않음. 엑셀은 저장 시 구분자를 쉼표로 바꾸고 유효숫자를 표시값 기준으로 잘라냄

In [1]:
import pandas as pd
import csv

filepath = r'C:\data\2023년 제7차 근로환경조사 원시자료.csv'

# 구분자는 탭, 인코딩은 cp949, 행 전체를 감싼 따옴표는 무시(QUOTE_NONE)
df = pd.read_csv(filepath, sep='\t', encoding='cp949',
                 quoting=csv.QUOTE_NONE, dtype=str)

# 첫/끝 컬럼에 붙은 따옴표와 꼬리 쉼표 제거
df.columns = [c.strip().strip('"').rstrip(',').strip('"') for c in df.columns]
first, last = df.columns[0], df.columns[-1]
df[first] = df[first].str.lstrip('"')
df[last]  = df[last].str.replace(r'",{0,3}$', '', regex=True)
df = df.replace(' ', pd.NA)

print(f'행 : {df.shape[0]}, 열 : {df.shape[1]}')   # 50195, 439
print(df.columns[:8].tolist())

행 : 50195, 열 : 439
['id', 'wt1', 'wt2', 'wt3', 'area', 'gender', 'year', 'age']


In [2]:
# 눈으로 보기 (엑셀 대신)
pd.set_option('display.max_columns', 30)
print(df.iloc[:10, :15])

          id          wt1               wt2               wt3 area gender  \
0  1000019_1  897.1816327  675.720677270798  1.18184603629422    1      1   
1  1000019_2  897.1816327  501.450045059828  .877043974068019    1      2   
2  1000042_1  897.1816327   568.48965743281  .994297304952782    1      1   
3  1000042_2  897.1816327  555.526955700343  .971625336819356    1      2   
4  1000049_1  897.1816327  387.349267596592  .677479929161676    1      1   
5  1000051_1  897.1816327  569.548473872926  .996149191471881    1      1   
6  1000053_1  897.1816327  331.675803102638  .580106168742357    1      1   
7  1000056_1  897.1816327  692.288622169203  1.21082363112942    1      1   
8  1000057_1  897.1816327    414.1639293313  .724379192054935    1      1   
9  1000057_2  897.1816327  534.554037506525  .934943374774034    1      2   

   year age estat country country_etc emp_type selfemp_be selfemp_be_etc  \
0  1989  34     2       1        <NA>        3       <NA>           <NA>   


### 1.2 파일 무결성 검증
- 5만 행 규모에서는 데이터 오염을 눈으로 확인할 수 없으므로 수치로 대조
- 코드북에 변수별 빈도가 기재되어 있어 이를 정답지로 사용
- 검증 항목: emp_type, gender, satisfaction의 값별 빈도 및 wt3의 분포
- 하나라도 어긋나면 파일 손상으로 판단하고 재다운로드

In [3]:
# 코드북 기재값과 대조 — 하나라도 어긋나면 파일이 오염된 것
expected = {
    'emp_type'    : {'1': 12867, '2': 3138, '3': 30150, '4': 4040},
    'gender'      : {'1': 23678, '2': 26517},
    'satisfaction': {'1': 2379, '2': 37688, '3': 8458,
                     '4': 1372, '8': 275, '9': 23},
}

for var, exp in expected.items():
    got = df[var].value_counts().sort_index().to_dict()
    print(f'{var:13s} {"통과" if got == exp else "불일치"}')

w = pd.to_numeric(df['wt3'], errors='coerce')
print(f'wt3 평균 {w.mean():.4f} (기대 1.0000), 최대 {w.max():.4f} (기대 20.7537)')

emp_type      통과
gender        통과
satisfaction  통과
wt3 평균 1.0000 (기대 1.0000), 최대 20.7537 (기대 20.7537)


> **검증 통과** <br>
> 세 변수의 빈도와 가중치 분포가 코드북 기재값과 일치. 원본 무결성 확인됨.

### 1.3 분석 모집단 확정 및 변수 축소
- 이 조사는 종사상지위(emp_type)에 따라 문항이 분기됨
- 임금근로자(emp_type=3)로 한정하면 공통 문항에 더해 Q11~Q26(고용형태, 상사 자질, 사업장 평가, 노조 유무)을 사용할 수 있음
- 판단이 필요 없는 변수부터 기계적으로 제거: 주관식 기타(_etc), 동거 가구원 정보(hm_), 타 지위 전용 문항(selfemp_/semp_/unfw_), 결측률 90% 초과 문항
- 설계 정보(id, wt1~wt3, stratification, district, household)는 복합표본 분석에 필요하므로 보존

### 1.4 결측코드 처리
- 무응답 유형별 코드: 7/77/777 = 해당없음, 8/88/888 = 모름·무응답, 9/99/999 = 거절
- 주의: 코드의 자릿수가 변수마다 다름. 유효값 범위와 겹치지 않도록 설계되어 있음
- 일괄 치환 금지. age의 77·88은 실제 나이(383명), ctime의 7·8·9는 실제 통근시간(분, 559명)
- 검산 기준: 전체 표본에서 ctime의 777/888/999를 제거하면 코드북 기재값(유효 47,335 / 평균 38.17 / 표준편차 32.361)과 일치해야 함

In [4]:
import pandas as pd
import numpy as np

# --- 01-1. 모집단 한정 ---
emp = df[df['emp_type'] == '3'].copy()
print(f'임금근로자 : {len(emp):,}명')

# --- 01-2. 기계적 축소 ---
drop_cols = (
    [c for c in emp.columns if c.endswith('_etc')]
  + [c for c in emp.columns if c.startswith('hm_')]
  + [c for c in emp.columns if c.startswith(('selfemp_', 'semp_', 'unfw_'))]
)
emp = emp.drop(columns=drop_cols)

# 임금근로자에게 전원 결측이거나 결측률 90% 넘는 열 제거
na_ratio = emp.isna().mean()
emp = emp.drop(columns=na_ratio[na_ratio > 0.90].index)
print(f'변수 : 439 → {emp.shape[1]}개')

# --- 01-3. 결측코드 처리 시범 (3개만) ---
emp['satisfaction'] = pd.to_numeric(emp['satisfaction'], errors='coerce')
emp.loc[emp['satisfaction'].isin([8, 9]), 'satisfaction'] = np.nan

emp['ctime'] = pd.to_numeric(emp['ctime'], errors='coerce')
emp.loc[emp['ctime'].isin([777, 888, 999]), 'ctime'] = np.nan

emp['age'] = pd.to_numeric(emp['age'], errors='coerce')   # 결측코드 없음, 그대로

print(f"\nsatisfaction 유효 : {emp['satisfaction'].notna().sum():,}")
print(f"ctime  평균 {emp['ctime'].mean():.2f} / 표준편차 {emp['ctime'].std():.3f}")
print(f"age    평균 {emp['age'].mean():.2f} / 최대 {emp['age'].max():.0f}")

임금근로자 : 30,150명
변수 : 439 → 291개

satisfaction 유효 : 30,012
ctime  평균 43.33 / 표준편차 29.674
age    평균 48.23 / 최대 93


> **가중치 재표준화 필요** <br>
> wt3는 전체 50,195명 기준으로 평균 1.0이 되도록 표준화된 값. 임금근로자만 추출하면 평균이 1.2802로 틀어지므로 부분집합 기준으로 재표준화해야 함.

### 1.5 결측코드 처리 — 변수군별 분기

설문 원시자료의 무응답은 유형별 숫자 코드로 기록되어 있다.
그러나 같은 숫자라도 변수에 따라 유효값일 수도, 결측일 수도 있으므로
일괄 치환이 불가능하다. 코드북의 '유효한 값 / 결측값' 구분선이 유일한 판단 근거다.


In [5]:
import numpy as np

KEEP_STR = ['id']                                   # 숫자 변환 금지

# A. 노출시간 척도 — 7=전혀 없음(유효), 결측은 8·9뿐
SCALE7 = ([f'hazard_phy{i}' for i in range(1,10)]
        + [f'hazard_erg{i}' for i in range(1,7)]
        + [f'hazard_psy{i}' for i in range(1,4)]
        + [f'useequip{i}'   for i in range(1,4)]
        + ['winten2_1','winten2_2'])

# D. 연속형·코드 변수 — 변수별 개별 지정, 빈 리스트는 '결측코드 없음'
NUMERIC = {
    'age':[], 'year':[], 'area':[], 'ind':[], 'ind2':[99], 'occ':[], 'occ2':[999],
    'wday_week':[], 'hh_num':[], 'eli_num':[], 'target':[], 'mode':[],
    'wtime_con2_day':[], 'wduration_y':[], 'comp_emp':[],
    'wtime_week':[], 'wtime_r':[], 'wtime_con_r':[], 'ptime_week':[],
    'edu':[8,9],                                    # B. 7=대학원 이상(유효)
    'earning2':[77,88,99], 'earning2_r':[77,88,99],
    'ctime':[777,888,999], 'earning1':[7777,8888,9999], 'earning1_r':[7777,8888,9999],
    'comp_size2':[88,99], 'comp_sizea_r':[88,99], 'comp_sizeb_r':[88,99],
    'emp_con_period_y':[77,88,99], 'emp_con_period_m':[77,88,99],
    'emp_con_period_r':[777,888,999],
    'heal_abs1':[888,999], 'job_c1':[666,888,999], 'job_c1_r':[888,999],
    'ptime_r':[888,999], 'woutside3_1':[888,999], 'woutside3_2':[888,999],
    'wt1':[], 'wt2':[], 'wt3':[],
    'stratification':[], 'district':[], 'household':[],
}

for c in emp.columns:
    if c in KEEP_STR:
        continue                                     # 문자열 그대로 보존
    emp[c] = pd.to_numeric(emp[c], errors='coerce')   # 컬럼 단위로 변환
    if   c in NUMERIC: codes = NUMERIC[c]
    elif c in SCALE7:  codes = [8, 9]
    else:              codes = [7, 8, 9]              # C. 일반 범주형
    if codes:
        emp.loc[emp[c].isin(codes), c] = np.nan

# --- 검산 ---
for c in ['id','hazard_phy1','occ','ind','area','edu','wsituation1','satisfaction']:
    s = emp[c]
    tail = '' if c == 'id' else f' / 최대 {s.max():.0f}'
    print(f'{c:14s} 유효 {s.notna().sum():6,}{tail}')

id             유효 30,150
hazard_phy1    유효 30,075 / 최대 7
occ            유효 30,150 / 최대 10
ind            유효 30,150 / 최대 21
area           유효 30,150 / 최대 17
edu            유효 30,123 / 최대 7
wsituation1    유효 28,074 / 최대 5
satisfaction   유효 30,012 / 최대 4


- **A. 노출시간 척도** (hazard_phy·erg·psy, useequip, winten2)
  - 1=근무시간 내내 ~ 7=전혀 없음. **7이 척도의 끝점이므로 유효값**
  - 결측은 8(모름/무응답), 9(거절)뿐
  - 7을 결측 처리하면 위험에 노출되지 않은 응답자가 전부 소실됨

- **B. 7 이상이 유효 범주인 변수** (edu, occ, ind, wday_week 등)
  - edu: 7=대학원 재학 이상 / occ: 8·9·10=장치조작·단순노무·군인
  - 범주 자체가 7을 넘어가므로 자릿수 기준 치환이 위험

- **C. 일반 범주형** (wsituation, wstat, emp_comp_ass, disc, wwa 등)
  - 유효값이 1-5 또는 1-2 범위. 7=해당없음, 8=모름/무응답, 9=거절
  - 7·8·9 모두 결측 처리

- **D. 연속형·코드 변수** (age, ctime, earning1, 사업장규모 등)
  - 결측코드의 자릿수가 유효값 범위와 겹치지 않게 설계되어 변수마다 다름
  - 예: age는 결측코드 없음(77·88은 실제 나이) / ctime은 777·888·999 / earning1은 7777·8888·9999
  - 변수별 개별 지정이 필요하며, 미확인 변수는 **손대지 않는 것이 기본값**

> **기본값 설계 원칙** <br>
> 초안에서는 미분류 변수에 [7,8,9]를 자동 적용했으나, 이로 인해 산업·직업 분류코드와
> 근속연수 등 연속형 변수 약 5만 셀이 조용히 소실되었다. 오류 메시지는 발생하지 않았다.
> 안전한 기본값은 '일괄 치환'이 아니라 '아무것도 하지 않음'이며,
> 처리 대상을 명시적으로 나열하는 화이트리스트 방식으로 전환했다.

> **검산 해석** <br>
> hazard_phy1 최대 7 — A그룹 분기가 작동, 노출 없음 응답이 보존됨 <br>
> occ 최대 10, ind 최대 21 — 분류코드가 잘리지 않음 <br>
> edu 최대 7 — 대학원 이상 학력 보존 <br>
> wsituation1 최대 5 — C그룹 분기가 작동, 해당없음(7)이 결측 처리됨 <br>
> id 유효 30,150 — 문자열 식별자가 숫자 변환에서 보호됨

### 1.6 구조적 결측 정리

결측률이 80%를 넘는 변수가 다수 존재하나, 이는 데이터 품질 문제가 아니라
설문의 분기(skip pattern) 구조가 반영된 결과다. 기계적으로 삭제하면
정보가 아니라 응답자를 잃는다.

In [6]:
import numpy as np

emp = emp.copy()                       # 조각난 블록 정리 (경고 해소)

# ① 누락됐던 결측코드 8개
EXTRA = {'wduration':[88,99], 'wday':[88,99], 'wtime':[888,999], 'ptime':[888,999],
         'wtime_night_a':[88,99], 'wtime_sun_a':[88,99],
         'wtime_sat_a':[88,99],   'wtime_long_a':[88,99]}
for c, codes in EXTRA.items():
    emp.loc[emp[c].isin(codes), c] = np.nan

# ② '안 함' 응답자의 일수는 결측이 아니라 0
for a, b in [('wtime_sun_a','wtime_sun'), ('wtime_sat_a','wtime_sat')]:
    emp.loc[(emp[a] == 0) & emp[b].isna(), b] = 0

# ③ 통합·계산 변수(_r)가 있으면 원본은 제거
REDUNDANT = ['earning1','earning2','earning2_r','wtime','wtime_week','wtime_month',
             'ptime','ptime_week','ptime_month','job_c1','wduration','wduration_y',
             'emp_con_period_y','emp_con_period_m','wtime_con1','wtime_con2',
             'wtime_con2_day','wtime_con2_week','wtime_con2_month','wday',
             'comp_size2','comp_size3','comp_size4']

# ④ 본문항이 있는 후속문항 제거
FOLLOWUP = ['ch_ic_a','ch_me_a','ch_ps_a','emp_suggest1_1','emp_suggest2_1','heal_lim2',
            'emp_con_renew','emp_tra1_1','emp_tra2_1','emp_tra_ass1','emp_tra_ask',
            'wteam2','wteam3_1','wteam3_2','wteam3_3','job_c2','job_c3','wtime_con3',
            'woutside3_1','woutside3_2','winterrupt2','heal_wsick2','emp_expect',
            'emp_expect_mon','emp_keep2','emp_expect2','wplace_sl',
            'wtime_night2_hours','wtime_night3_times']

# ⑤ 다문화가구·외국인 한정 문항 제거
FOREIGN = ['disc2','disc3','disc4']

drop = [c for c in REDUNDANT + FOLLOWUP + FOREIGN if c in emp.columns]
emp = emp.drop(columns=drop)
print(f'변수 : 291 → {emp.shape[1]}개')

# ⑥ 종속변수 결측 제거 → 그 다음에 가중치 재표준화 (순서 중요)
emp = emp[emp['satisfaction'].notna()].copy()
emp['wt_std'] = emp['wt3'] / emp['wt3'].mean()

print(f"최종 : {len(emp):,}명 × {emp.shape[1]}개 / wt_std 평균 {emp['wt_std'].mean():.4f}")
na = emp.isna().mean()
print(f'결측률 50% 초과 : {(na>0.5).sum()}개')

emp.to_csv(r'C:\data\kwcs2023_clean.csv', index=False, encoding='utf-8')
print(f'저장 완료 : {len(emp):,}행 × {emp.shape[1]}열')

변수 : 291 → 249개
최종 : 30,012명 × 250개 / wt_std 평균 1.0000
결측률 50% 초과 : 7개
저장 완료 : 30,012행 × 250열


- **본문항 / 후속문항 구조**
  - "변화가 있었는가"(본문항, 결측 2.6%) → "그 영향은"(후속문항, 결측 89.7%)
  - 후속문항은 본문항에서 특정 응답을 한 사람에게만 제시됨
  - 본문항을 유지하고 후속문항을 제거하면 정보 손실 없이 결측률이 정리됨

- **결측이 실제로는 0인 경우**
  - wtime_sun_a(일요일 근무 여부)=0인 26,656명은 wtime_sun(근무 일수)이 결측
  - 근무하지 않았으므로 일수를 묻지 않은 것. 결측이 아니라 0일에 해당
  - 0으로 복원하면 결측률 88.9% → 0.5%

- **통합·계산 변수 우선 사용**
  - 응답 단위(주/월)를 응답자가 선택하는 문항은 원본이 여러 컬럼으로 분리됨
  - 조사기관이 단위를 통일한 _r 변수(wtime_r, ptime_r, earning1_r 등)가 별도 제공됨
  - 예: earning1 결측 22.1% / earning2 81.2% vs earning1_r 3.4%
  - 원본 컬럼은 제거하고 _r 변수만 사용

- **누락 결측코드 보완**
  - wduration, wday, wtime, ptime 및 근무여부(_a) 변수 8개에서 88/99, 888/999 미처리 확인
  - ptime의 777은 '현재와 동일'을 뜻하는 유효 응답이므로 결측 처리 대상이 아님

> **가중치 재표준화 순서** <br>
> wt3는 전체 50,195명 기준 평균 1.0으로 표준화된 값이므로,
> 임금근로자 부분집합에서는 평균이 1.2802로 틀어진다.
> 재표준화는 분석 대상 행이 최종 확정된 뒤에 수행해야 평균이 정확히 1.0이 된다.

> **PerformanceWarning 대응** <br>
> 반복문으로 컬럼을 개별 대입하면 내부 메모리 블록이 조각나 경고가 발생한다.
> 계산 결과에는 영향이 없으며, 루프 종료 후 `emp = emp.copy()`로 해소한다.

## 단계: 02. 탐색적 데이터 분석 (EDA)
- 목표: 250개 변수의 구조를 파악하고 모델 투입 변수를 단계적으로 선별
- 절차: 데이터 탐색 → 1차 변수 선택 → 파생변수 생성 → 2차 변수 선택 → 최종 선택

### 2.1 종속변수 분포 — 가중 vs 비가중

- 종속변수: satisfaction (Q78 근로환경 만족도, 4점 순서형, 값이 클수록 불만족)
- 복합표본 설계이므로 단순 빈도가 아니라 가중 추정치가 모집단을 대표함
- Python(wt_std 적용)과 SAS PROC SURVEYFREQ 결과가 소수점 둘째 자리까지 일치

> **불만족 비율 17.81% → 19.02%** <br>
> 가중치 적용 시 불만족(3+4)이 1.21%p 상승. 표본에서 상대적으로 적게 추출된 집단이
> 더 불만족스럽다는 의미이며, 보고서에 사용할 값은 가중 추정치인 19.02%다.

> **모집단 복원 검증** <br>
> SAS의 Sum of Weights = 21,944,399. 통계청 경제활동인구조사 기준 2023년
> 임금근로자 약 2,195만 명과 0.03% 차이로 일치. 가중치 적용이 정상 작동함을 확인.

> **설계효과(Design Effect)** <br>
> 조사구 4,831개에서 평균 6.21명씩 추출된 집락표본이며, 가중치 변동계수는 0.786이다.
> 설계효과가 2.59~3.71(평균 약 2.9)로, 표본 30,012명의 실질적 정밀도는
> 단순임의추출 약 10,300명 수준이다. 층화·집락을 무시하면 표준오차가
> 약 1.7배 과소추정되어 유의성이 과장된다.

In [7]:
# --- 2.1 종속변수 분포: 가중 vs 비가중 ---
s, w = emp['satisfaction'], emp['wt_std']
unw = s.value_counts(normalize=True).sort_index() * 100
wtd = s.groupby(s).apply(lambda g: w[g.index].sum()) / w.sum() * 100

lab = {1:'매우 만족', 2:'만족', 3:'별로 만족않음', 4:'전혀 만족않음'}
for k in [1, 2, 3, 4]:
    print(f'{k} {lab[k]:12s} 비가중 {unw[k]:6.2f}% 가중 {wtd[k]:6.2f}% ({wtd[k]-unw[k]:+.2f})')
print(f'불만족(3+4) : {unw[3]+unw[4]:.2f}% -> {wtd[3]+wtd[4]:.2f}%')

1 매우 만족        비가중   5.35% 가중   5.44% (+0.09)
2 만족           비가중  76.84% 가중  75.54% (-1.30)
3 별로 만족않음      비가중  15.14% 가중  16.18% (+1.03)
4 전혀 만족않음      비가중   2.66% 가중   2.84% (+0.18)
불만족(3+4) : 17.81% -> 19.02%


### 2.2 변수 유형 분류 및 이상치 점검

- 설계정보 13개를 제외한 분석 대상 237개를 고유값 개수 기준으로 분류
- 순서형/소범주 118, 이분형 100, 연속형 13, 명목형 다범주 6
- 연속형이 13개뿐이라는 점이 앞선 두 프로젝트(HIRA/NFA)와 결정적으로 다름
- 주된 기법이 비모수 검정이 아니라 범주형 자료분석이 됨

> **저분산 변수 25개는 보류** <br>
> 한 값에 95% 이상 몰린 변수가 25개 존재하나 즉시 제거하지 않는다.
> asb5(신체적 폭력)는 99.7%가 '없음'이지만 나머지 90명은 실제 피해자이며,
> 개별로는 검정력이 부족해도 합산 지수로 묶으면 유효한 분산을 갖는다.

> **연속형 이상치 점검에서 결측코드 추가 발견** <br>
> comp_emp(부하 직원 수)의 최대값이 9,999,999로 나타나 확인한 결과
> 8888888(모름/무응답), 9999999(거절)의 7자리 결측코드였다.
> 처리 후 평균 660,440 → 1.50명. 이 조사의 결측코드는 1자리부터 7자리까지 존재하므로
> 연속형 변수는 반드시 분포를 확인한 뒤 사용해야 한다.
> heal_abs1의 최대 180일(결근)은 실제 값이므로 유지한다.

In [8]:
# --- 2.2 변수 유형 분류 ---
DESIGN = ['id','wt1','wt2','wt3','wt_std','stratification','district','household',
          'target','mode','emp_type','estat','country']
analysis = [c for c in emp.columns if c not in DESIGN]

rows = []
for c in analysis:
    nu = emp[c].dropna().nunique()
    t = ('이분형' if nu <= 2 else '순서형/소범주' if nu <= 7
    else '명목형(다범주)' if nu <= 25 else '연속형')
    rows.append((c, t, nu, round(emp[c].isna().mean(), 3)))

vtype = pd.DataFrame(rows, columns=['변수', '유형', '고유값', '결측률'])
print(f'\n분석 대상 {len(analysis)}개')
print(vtype['유형'].value_counts().to_string())

# --- 저분산 변수 탐지 (버리지 말고 목록만) ---
lowvar = [(c, round(emp[c].dropna().value_counts(normalize=True).iloc[0], 3))
          for c in analysis if len(emp[c].dropna()) > 0
          and emp[c].dropna().value_counts(normalize=True).iloc[0] > 0.95]
print(f'\n한 값에 95% 이상 몰린 변수 {len(lowvar)}개')

# 결측코드 보완 : comp_emp는 7자리 코드 사용 (8888888=모름, 9999999=거절)
emp.loc[emp['comp_emp'].isin([8888888, 9999999]), 'comp_emp'] = np.nan
print(f"comp_emp 유효 {emp['comp_emp'].notna().sum():,} / 평균 {emp['comp_emp'].mean():.2f} / 최대 {emp['comp_emp'].max():.0f}")

# 정제본 다시 저장 (SAS가 읽을 파일이므로 반드시 갱신)
emp.to_csv(r'C:\data\kwcs2023_clean.csv', index=False, encoding='utf-8')
print('저장 갱신 완료')


분석 대상 237개
유형
순서형/소범주     118
이분형         100
연속형          13
명목형(다범주)      6

한 값에 95% 이상 몰린 변수 25개
comp_emp 유효 27,801 / 평균 1.50 / 최대 1000
저장 갱신 완료


In [9]:
from scipy.stats import spearmanr

DESIGN = ['id','wt1','wt2','wt3','wt_std','stratification','district','household',
          'target','mode','emp_type','estat','country','panel_survey','year']
cand = [c for c in emp.columns if c not in DESIGN and c != 'satisfaction']

# --- 2.3-1 종속변수와의 단변량 상관 (Spearman) ---
y = emp['satisfaction']
rows = []
for c in cand:
    m = emp[c].notna() & y.notna()
    if m.sum() < 1000:
        continue
    rho, p = spearmanr(emp.loc[m, c], y[m])
    rows.append((c, rho, p, int(m.sum())))

corr = (pd.DataFrame(rows, columns=['변수','rho','p','n'])
        .assign(절대값=lambda t: t.rho.abs())
        .sort_values('절대값', ascending=False))

print(f'검토 {len(corr)}개')
print(corr.head(20)[['변수','rho','n']].to_string(index=False))
print(f"\n|rho| < 0.05 : {(corr.절대값 < 0.05).sum()}개")

검토 234개
           변수       rho     n
     wbalance  0.281355 29948
        weng1  0.276040 29982
        weng2  0.245962 29988
       wstat1  0.245073 29460
 wsituation11  0.244141 29047
   safeinform  0.239831 29451
emp_manaqual1  0.239416 28193
  wsituation8  0.234119 29957
    heal_cond  0.232844 29980
  wsituation7  0.230178 29947
       wstat3  0.229270 29258
emp_comp_ass1  0.225529 28784
         who1  0.218541 29991
  wsituation1  0.217541 27967
  wsituation2  0.217006 28968
   income_bal  0.214095 29928
       wstat2  0.209013 29002
emp_comp_ass6  0.208732 28531
    heal_risk -0.208458 29788
emp_manaqual5  0.203759 28082

|rho| < 0.05 : 81개


In [10]:
# --- 2.3-2 문항 블록별 신뢰도 (Cronbach's alpha) ---
def cronbach(df):
    d = df.dropna()
    k = d.shape[1]
    return k / (k-1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1)), len(d)

BLOCKS = {
    '업무상황 wsituation' : [f'wsituation{i}' for i in range(1, 15)],
    '업무동의 wstat'      : [f'wstat{i}' for i in range(1, 8)],
    '직무열의 weng'       : [f'weng{i}' for i in range(1, 6)],
    '상사자질 manaqual'   : [f'emp_manaqual{i}' for i in range(1, 6)],
    '사업장평가 comp_ass' : [f'emp_comp_ass{i}' for i in range(1, 7)],
    '물리위험 hazard_phy' : [f'hazard_phy{i}' for i in range(1, 10)],
    '인간공학 hazard_erg' : [f'hazard_erg{i}' for i in range(1, 7)],
    '감정위험 hazard_psy' : [f'hazard_psy{i}' for i in range(1, 4)],
    'WHO-5 who'          : [f'who{i}' for i in range(1, 6)],
    '일가정갈등 wwa'      : [f'wwa{i}' for i in range(1, 6)],
    '수면 sleep'          : [f'sleep{i}' for i in range(1, 4)],
    '기술우려 imte'       : [f'imte{i}' for i in range(1, 6)],
    '작업특성 condim'     : [f'condim{i}' for i in range(1, 7)],
    '근무형태 length'     : [f'wtime_length{i}' for i in range(1, 6)],
    '자율성 decla'        : [f'decla{i}' for i in range(1, 4)],
    '폭력 asb'            : ['asb1','asb2','asb3','asb4','asb5','asb6','asb7'],
    '차별 disc'           : ['disc1','disc5','disc6','disc7','disc8','disc9','disc10','disc11'],
    '건강문제 heal_prob'  : [f'heal_prob{i}' for i in [1,2,3,4,5,6,8]],
}

print(f"\n{'블록':22}{'문항':>4s}{'alpha':>8s}{'완전응답':>9s} 판정")
for name, items in BLOCKS.items():
    items = [c for c in items if c in emp.columns]
    a, n = cronbach(emp[items])
    verdict = '우수' if a >= .8 else '양호' if a >= .7 else '수용가능' if a >= .6 else '낮음'
    print(f'{name:22s}{len(items):4d}{a:8.3f}{n:9,}  {verdict}')


블록                      문항   alpha     완전응답 판정
업무상황 wsituation         14   0.875   25,827  우수
업무동의 wstat               7   0.707   27,511  양호
직무열의 weng                5   0.732   29,968  양호
상사자질 manaqual            5   0.860   27,901  우수
사업장평가 comp_ass           6   0.862   27,814  우수
물리위험 hazard_phy          9   0.915   29,831  우수
인간공학 hazard_erg          6   0.410   29,898  낮음
감정위험 hazard_psy          3   0.695   29,926  수용가능
WHO-5 who                5   0.927   29,970  우수
일가정갈등 wwa                5   0.896   28,915  우수
수면 sleep                 3   0.879   30,000  우수
기술우려 imte                5   0.895   29,638  우수
작업특성 condim              6   0.579   29,675  낮음
근무형태 length              5   0.781   29,916  양호
자율성 decla                3   0.907   29,741  우수
폭력 asb                   7   0.566   29,914  낮음
차별 disc                  8   0.624   17,047  수용가능
건강문제 heal_prob           7   0.712   29,920  양호


In [11]:
# --- 2.3-3 항목-총점 상관 진단 ---
#     SAS의 PROC CORR ALPHA가 wsituation12, wsituation14도
#     척도를 해치고 있음을 알려주었으므로 진단 대상에 추가한다.
for name in ['인간공학 hazard_erg', '작업특성 condim', '업무상황 wsituation']:
    items = BLOCKS[name]
    total = emp[items].sum(axis=1)
    print(f'\n[{name}] 항목-총점 상관')
    for c in items:
        print(f'   {c:16s} {emp[c].corr(total - emp[c]):+.3f}')


[인간공학 hazard_erg] 항목-총점 상관
   hazard_erg1      +0.626
   hazard_erg2      +0.305
   hazard_erg3      +0.487
   hazard_erg4      +0.066
   hazard_erg5      -0.374
   hazard_erg6      +0.486

[작업특성 condim] 항목-총점 상관
   condim1          +0.501
   condim2          +0.576
   condim3          +0.528
   condim4          -0.192
   condim5          +0.235
   condim6          +0.380

[업무상황 wsituation] 항목-총점 상관
   wsituation1      +0.553
   wsituation2      +0.529
   wsituation3      +0.623
   wsituation4      +0.613
   wsituation5      +0.559
   wsituation6      +0.449
   wsituation7      +0.502
   wsituation8      +0.527
   wsituation9      +0.559
   wsituation10     +0.429
   wsituation11     +0.561
   wsituation12     +0.260
   wsituation13     +0.484
   wsituation14     +0.248


### 2.3 종속변수 상관 및 문항 신뢰도

#### 2.3-1 단변량 순위상관 (Spearman)

- 종속변수가 순서형이므로 Pearson이 아닌 Spearman을 사용
- 검토 대상 234개 변수 중 최대 상관은 rho = 0.281 (wbalance)
- Python과 SAS PROC CORR 결과가 소수점 다섯째 자리까지 일치

> **척도 방향이 통일되어 있다** <br>
> 이 조사의 리커트 문항은 대부분 1=긍정, 값이 클수록 부정이며 종속변수도 동일하다.
> 따라서 양(+)의 상관은 '부정적 경험이 많을수록 불만족'으로 자연스럽게 읽힌다.
> heal_risk(1=위험함, 2=아니다)와 hazard_erg3(1=내내 노출, 7=전혀 없음)만
> 음(-)의 부호로 나타나지만 방향을 따지면 같은 해석이다.

> **압도적 단일 요인이 없다** <br>
> 최대 상관이 0.281이고 0.20~0.28 구간에 20개 가까운 변수가 몰려 있다.
> 근로환경 만족도는 하나의 결정적 요인이 아니라 여러 요인이 조금씩 기여하는 구조다.
> 단변량 상관이 낮다고 제거하면 다변량에서 유의한 변수를 잃을 수 있으므로,
> |rho| < 0.05인 81개도 파생변수 재료 여부를 확인한 뒤 처리한다.

#### 2.3-2 신뢰도 분석 (Cronbach's alpha)

- alpha는 문항들이 하나의 개념을 일관되게 측정하는지 판단하는 지표
- 18개 문항 블록 중 9개가 alpha >= 0.8 (우수), 4개가 0.7~0.8 (양호)

> **alpha가 낮을 때는 버리지 말고 진단한다** <br>
> hazard_erg(0.410), condim(0.579)의 항목-총점 상관을 확인한 결과
> hazard_erg5(앉아 있는 자세) -0.374, condim4(단조롭다) -0.192로 혼자 음수였다.
> 나머지 문항과 방향이 반대인 역방향 문항이 척도를 무너뜨린 것이다.
> 제외 시 alpha는 각각 0.410 → 0.692, 0.579 → 0.733으로 회복된다.

> **SAS 출력에서 추가 발견** <br>
> PROC CORR의 '변수를 제외했을 때의 alpha' 표에서 wsituation12(스트레스),
> wsituation14(감정 억제)를 제외하면 alpha가 0.8747 → 0.8753, 0.8767로 오히려 상승했다.
> 나머지 12문항이 '자원'을 묻는 반면 이 둘은 '부담'을 묻는 다른 개념이다.
> Python 코드에서는 이 블록을 진단 대상에 넣지 않아 놓쳤던 부분이다.

> **alpha는 형성지표에 부적절하다** <br>
> asb(폭력 7종, alpha=0.566)는 언어폭력·성희롱·위협 등 서로 독립적으로 발생하는
> 사건을 묻는다. 하나의 잠재개념이 문항에 반영되는 구조(반영지표)가 아니라
> 여러 사건이 합쳐져 개념을 구성하는 구조(형성지표)이므로 내적 일관성이 낮은 것이 정상이다.
> 평균이 아니라 경험 개수 합산으로 지표화한다.

#### 2.3-3 차원 확인 (주성분분석)

> **alpha가 높다고 1차원인 것은 아니다** <br>
> alpha는 문항 수가 늘면 상관이 중간 수준이어도 자동으로 상승한다.
> alpha는 '묶을 만한가'를 알려줄 뿐 '몇 개로 묶을 것인가'는 답하지 않는다.

#### 추가

> wsituation(alpha=0.875) → 고유값 5.48 / 1.58 / 1.25로 3개 요인
>   F1 지지·공정(1,2,7,8,9,10,11) / F2 참여·영향력(3,4,5,6,13) / F3 부담(12,14)
>   신뢰도 진단이 지목한 12·14번이 요인분석에서도 별도 요인으로 분리되었다.
>
> hazard_phy(alpha=0.915) → 고유값 5.67 / 1.01로 2개 요인
>   F1 화학·생물(6,7,8,9) / F2 물리(1,2,3,4,5)
>
> 따라서 2.3 파생변수 단계에서는 블록당 지수 1개가 아니라
> 요인 구조에 맞춰 복수의 지수를 생성한다.

In [14]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', '{:.2f}'.format)


# ── 베리맥스 회전 ────────────────────────────────────────────
# 회전은 요인의 설명력 총량을 바꾸지 않는다. 축을 돌려서
# 각 문항의 적재량이 한 요인에는 크고 나머지에는 0에 가깝도록 만들어
# "이 문항은 어느 요인 소속인가"를 읽기 쉽게 하는 작업이다.
def varimax(L, kaiser=True, max_iter=100, tol=1e-6):
    """베리맥스 직교회전.
    kaiser=True : SAS PROC FACTOR의 기본값(NORM=KAISER)과 동일.
                  회전 전에 각 행을 공통성으로 나눠 길이를 1로 맞추고,
                  회전 후 되돌린다. 공통성이 큰 문항이 회전 방향을
                  독점하는 것을 막는다."""
    L = L.copy()
    h = np.sqrt((L**2).sum(axis=1)) if kaiser else np.ones(L.shape[0])
    L = L / h[:, None]                       # 정규화

    p, k = L.shape
    R, d_prev = np.eye(k), 0
    for _ in range(max_iter):
        Lam = L @ R
        u, s, vh = np.linalg.svd(
            L.T @ (Lam**3 - Lam @ np.diag(np.diag(Lam.T @ Lam)) / p))
        R, d = u @ vh, s.sum()
        if d_prev != 0 and d / d_prev < 1 + tol:
            break
        d_prev = d
    return (L @ R) * h[:, None]              # 원래 크기로 복원


# ── 블록 단위 요인분석 ───────────────────────────────────────
def factor_block(items, name, show_loadings=False):
    d = emp[[c for c in items if c in emp.columns]].dropna()
    cols = list(d.columns)

    R = np.corrcoef(d.T.values)                 # 문항 간 상관행렬
    ev, V = np.linalg.eigh(R)                   # 고유값 분해
    order = np.argsort(ev)[::-1]                # 큰 순서로 정렬
    ev, V = ev[order], V[:, order]

    k = int((ev > 1).sum())                     # Kaiser 기준: 고유값 1 초과
    print(f'{name:24s} 문항{len(cols):3d}  n={len(d):6,}  '
          f'고유값 {np.round(ev[:4], 2)}  요인 {k}개  '
          f'1요인 {ev[0]/len(cols)*100:.1f}%')

    if show_loadings and k >= 2:
        L = varimax(V[:, :k] * np.sqrt(ev[:k]))
        out = pd.DataFrame(L, index=cols, columns=[f'F{i+1}' for i in range(k)])
        out['주요인'] = out.abs().idxmax(axis=1)
        out['공통성'] = (L**2).sum(axis=1)          # ← 추가
        # 공통성 0.4 미만 = 요인 구조에 잘 맞지 않는 문항
        out['판정'] = np.where(out['공통성'] < 0.4, '검토필요', '')   # ← 추가
        print(out.round(2).to_string())
        print()
    return k


BLOCKS = {
    'wsituation 업무상황'  : [f'wsituation{i}' for i in range(1, 15)],
    'wstat 업무동의'       : [f'wstat{i}' for i in range(1, 8)],
    'weng 직무열의'        : [f'weng{i}' for i in range(1, 6)],
    'emp_manaqual 상사'    : [f'emp_manaqual{i}' for i in range(1, 6)],
    'emp_comp_ass 사업장'  : [f'emp_comp_ass{i}' for i in range(1, 7)],
    'hazard_phy 물리위험'  : [f'hazard_phy{i}' for i in range(1, 10)],
    'hazard_erg 인간공학'  : [f'hazard_erg{i}' for i in [1, 2, 3, 4, 6]],   # erg5 제외
    'hazard_psy 감정위험'  : [f'hazard_psy{i}' for i in range(1, 4)],
    'who WHO-5'           : [f'who{i}' for i in range(1, 6)],
    'wwa 일가정갈등'       : [f'wwa{i}' for i in range(1, 6)],
    'sleep 수면'           : [f'sleep{i}' for i in range(1, 4)],
    'imte 기술우려'        : [f'imte{i}' for i in range(1, 6)],
    'condim 작업특성'      : [f'condim{i}' for i in [1, 2, 3, 5, 6]],       # condim4 제외
    'wtime_length 근무형태': [f'wtime_length{i}' for i in range(1, 6)],
    'decla 자율성'         : [f'decla{i}' for i in range(1, 4)],
    'heal_prob 건강문제'   : [f'heal_prob{i}' for i in [1, 2, 3, 4, 5, 6, 8]],
}

print('=== 1단계 : 블록별 요인 수 ===')
n_factors = {name: factor_block(items, name) for name, items in BLOCKS.items()}

print('\n=== 2단계 : 2요인 이상 블록의 적재량 ===')
for name, k in n_factors.items():
    if k >= 2:
        factor_block(BLOCKS[name], name, show_loadings=True)

=== 1단계 : 블록별 요인 수 ===
wsituation 업무상황          문항 14  n=25,827  고유값 [5.48 1.58 1.25 0.74]  요인 3개  1요인 39.1%
wstat 업무동의               문항  7  n=27,511  고유값 [2.82 1.23 0.76 0.66]  요인 2개  1요인 40.3%
weng 직무열의                문항  5  n=29,968  고유값 [2.44 1.54 0.5  0.32]  요인 2개  1요인 48.7%
emp_manaqual 상사          문항  5  n=27,901  고유값 [3.21 0.51 0.46 0.44]  요인 1개  1요인 64.3%
emp_comp_ass 사업장         문항  6  n=27,814  고유값 [3.57 0.59 0.51 0.48]  요인 1개  1요인 59.5%
hazard_phy 물리위험          문항  9  n=29,831  고유값 [5.67 1.01 0.66 0.38]  요인 2개  1요인 63.0%
hazard_erg 인간공학          문항  5  n=29,903  고유값 [2.33 0.94 0.82 0.54]  요인 1개  1요인 46.5%
hazard_psy 감정위험          문항  3  n=29,926  고유값 [2.1  0.68 0.22]  요인 1개  1요인 70.0%
who WHO-5                문항  5  n=29,970  고유값 [3.88 0.38 0.29 0.24]  요인 1개  1요인 77.5%
wwa 일가정갈등                문항  5  n=28,915  고유값 [3.55 0.54 0.42 0.31]  요인 1개  1요인 70.9%
sleep 수면                 문항  3  n=30,000  고유값 [2.42 0.32 0.26]  요인 1개  1요인 80.7%
imte 기술우려                문항  5  n=29,638 

### 2.3 상관 및 신뢰도 분석

- 목표: 234개 후보 변수의 관계 구조를 파악하고 지수화 대상을 식별
- 절차: 단변량 상관 → 신뢰도(alpha) → 항목 진단 → 요인분석
- 종속변수와 문항 척도가 모두 순서형이므로 Pearson이 아닌 Spearman을 사용

#### 2.3-1 단변량 순위상관

- 최대 상관은 rho = 0.281 (wbalance, 근무시간이 개인생활에 부적당)
- 0.20~0.28 구간에 20개 가까운 변수가 몰려 있음
- Python과 SAS PROC CORR 결과가 소수점 다섯째 자리까지 일치

> **척도 방향이 통일되어 있다** <br>
> 이 조사의 리커트 문항은 대부분 1=긍정, 값이 클수록 부정이며 종속변수도 동일하다.
> 따라서 양(+)의 상관은 '부정적 경험이 많을수록 불만족'으로 자연스럽게 읽힌다.
> heal_risk(1=위험함, 2=아니다)와 hazard_erg3(1=내내 노출, 7=전혀 없음)만
> 음(-)의 부호이지만 방향을 따지면 같은 해석이다.

> **압도적 단일 요인이 없다** <br>
> 만족도를 결정하는 단일 변수는 존재하지 않고 여러 요인이 조금씩 기여하는 구조다.
> 단변량 상관이 낮다고 즉시 제거하면 다변량에서 유의한 변수를 잃을 수 있으므로,
> |rho| < 0.05인 81개도 파생변수 재료 여부를 확인한 뒤 2.4에서 처리한다.

#### 2.3-2 신뢰도 분석 (Cronbach's alpha)

- alpha 판정 기준: 0.8 이상 우수 / 0.7~0.8 양호 / 0.6~0.7 수용가능 / 0.6 미만 낮음
- 18개 문항 블록 중 9개가 우수, 4개가 양호

#### 2.3-3 항목-총점 상관 진단

> **alpha가 낮을 때는 버리지 말고 원인을 찾는다** <br>
> hazard_erg(0.410), condim(0.579)의 항목-총점 상관을 확인한 결과
> hazard_erg5(앉아 있는 자세) -0.378, condim4(단조롭다) -0.193으로 혼자 음수였다.
> 나머지 문항과 방향이 반대인 문항이 척도를 무너뜨린 것이며,
> 제외 시 alpha는 각각 0.692, 0.733으로 회복된다.

> **SAS 출력이 잡아준 추가 사례** <br>
> PROC CORR ALPHA의 '변수를 제외했을 때의 alpha' 표에서
> wsituation12(스트레스), wsituation14(감정 억제)를 제외하면
> alpha가 0.8747 → 0.8753, 0.8767로 오히려 상승했다.
> 나머지 12문항이 '자원'을 묻는 반면 이 둘은 '부담'을 측정한다.
> Python 코드에서는 진단 대상에 넣지 않아 놓쳤던 부분으로,
> 이후 진단 대상에 wsituation을 추가했다.

> **alpha가 부적절한 경우** <br>
> asb(폭력 7종, alpha=0.566)는 언어폭력·성희롱·위협 등 서로 독립적으로 발생하는
> 사건을 묻는다. 하나의 잠재개념이 문항에 반영되는 반영지표가 아니라
> 여러 사건이 합쳐져 개념을 구성하는 형성지표이므로 내적 일관성이 낮은 것이 정상이다.
> 평균이 아니라 경험 개수 합산으로 지표화한다.

#### 2.3-4 탐색적 요인분석

- 주성분 추출 + 베리맥스 직교회전, Kaiser 기준(고유값 > 1)으로 요인 수 결정
- Python 고유값과 SAS PROC FACTOR의 MINEIGEN 판정이 16개 블록 전부 일치
- 목록별 제거 표본 크기도 7개 블록 전부 일치 (예: wsituation 25,827명)

> **alpha가 높다고 1차원인 것은 아니다** <br>
> alpha는 문항 수가 늘면 상관이 중간 수준이어도 자동으로 상승한다.
> '묶을 만한가'는 답하지만 '몇 개로 묶을 것인가'는 답하지 않으므로
> 지수화 전에 요인 수를 반드시 확인해야 한다.

> **1요인 블록 (10개) — 블록당 지수 1개** <br>
> emp_manaqual(64.3%), emp_comp_ass(59.5%), hazard_psy(70.0%), who(77.5%),
> wwa(70.9%), sleep(80.7%), imte(71.4%), decla(84.4%),
> hazard_erg(erg5 제외, 46.5%), condim(condim4 제외, 48.5%)
> 괄호는 제1요인 설명력. WHO-5는 SAS에서 'Rotation not possible with 1 factor'로
> 1요인임이 확인되었다.

> **2요인 이상 블록 (6개) — 요인별 분리 생성** <br>
> wsituation 3요인 : 지지·공정 / 참여·영향력 / 부담(12,14) <br>
> hazard_phy 2요인 : 화학·생물(6,7,8,9) / 물리(1,2,3,4,5) <br>
> wstat 2요인      : 보상·인정(1,2,3,5) / 고용불안(4,6,7) <br>
> weng 2요인       : 직무열의(1,2,3) / 소진(4,5) <br>
> wtime_length 2요인: 근무 규칙성(1,2,3,4) / 교대근무(5) <br>
> heal_prob 2요인  : 근골격계(1,2,3) / 정신·피로(4,5,6,8)

> **weng — alpha만 봤으면 놓쳤을 사례** <br>
> alpha=0.732 '양호'였으나 고유값 2.44 / 1.54로 2요인이며 제1요인 설명력이 48.7%였다.
> 직무열의(에너지·열정·몰입)와 소진(기진맥진·진이 빠짐)은
> 한 축의 양끝이 아니라 별개 구성개념이다. 두 요인을 평균 내면
> 서로 상쇄되어 의미 없는 지수가 만들어진다.

> **판정이 경계에 걸친 블록 — 스크리 도표 병행 확인 필요** <br>
> hazard_phy의 두 번째 고유값 1.01, wtime_length는 정확히 1.00으로
> Kaiser 기준을 아슬아슬하게 통과했다. 기계적 기준만으로 결정하기 불안하므로
> SAS의 스크리 도표에서 고유값이 꺾이는 지점을 함께 확인한다.

> **wsituation의 표본 손실 13.9%** <br>
> 14문항 목록별 제거로 4,185명이 탈락해 25,827명만 분석에 사용되었다.
> 2.5 파생변수 생성 시 완전응답 조건을 걸면 표본의 14%를 잃게 되므로,
> 응답한 문항의 평균으로 지수를 구성할지 별도 판단이 필요하다.

> **Python과 SAS 적재량 미세 차이 — Kaiser 정규화** <br>
> 초기 Python 구현에서 SAS와 적재량이 최대 0.02 차이가 났다.
> 원인은 SAS PROC FACTOR의 ROTATE=VARIMAX 기본값이 NORM=KAISER이기 때문이다.
> Kaiser 정규화는 회전 전에 각 문항의 적재 벡터 길이를 1로 맞추어
> 공통성이 큰 문항이 회전 방향을 독점하지 않게 하고, 회전 후 원래 크기로 되돌린다.
> Python 함수에 이를 반영하자 최대 오차가 0.0204 → 0.0005로 줄어 일치했다.
> 요인 수와 문항 배정 결론은 정규화 여부와 무관하게 동일했다.

> **공통성(Communality) — SAS 출력에서만 확인되는 지표** <br>
> 공통성은 해당 문항의 분산 중 추출된 요인들이 설명하는 비율이다.
> 0.4 미만이면 그 문항의 분산 대부분이 요인으로 설명되지 않는 고유분산이므로
> 요인 구조에 잘 맞지 않는다고 판단한다.
>
> 0.4 미만 문항 3개
>   wsituation6 (원할 때 휴식 가능) 0.392  — F1 0.415 / F2 0.463 교차적재
>   heal_prob4  (두통, 눈의 피로)    0.356  — F1 0.358 / F2 0.477
>   heal_prob6  (전신 피로)          0.342  — F1 0.401 / F2 0.425
>
> 세 문항 모두 공통성이 낮으면서 동시에 교차적재를 보인다.
> 두 지표가 같은 결론을 가리키므로 지수 구성에서 제외를 검토한다.

> **교차적재라도 성격이 다른 경우** <br>
> hazard_phy5(분진 흡입)는 F1 0.538 / F2 0.618로 교차적재이나
> 공통성이 0.672로 양호하다. 요인 구조에 맞지 않는 것이 아니라
> 두 요인 모두와 실질적으로 관련된 문항이다.
> 분진은 물리적 자극이면서 화학적 노출의 성격을 함께 가지므로 내용상 타당하다.
> 제거 대상이 아니라 배정 판단이 필요한 사례로 구분한다.

> **스크리 도표 판독** <br>
> wsituation의 고유값 차이(Difference)는 3.899 → 0.330 → 0.505 → 0.023으로
> 4번째 이후 평탄해진다. 스크리 기준으로도 3요인이 지지되어 Kaiser 기준과 일치했다.
>
> hazard_phy는 4.665 → 0.351 → 0.273으로 2·3번 낙차가 이후와 비슷해
> 스크리만으로는 1요인으로 읽을 여지가 있다. 다만 회전 후 설명분산이
> 3.44 / 3.23으로 균형 있게 나뉘고 내용상 화학·생물 계열과 물리 계열이
> 뚜렷이 구분되므로 2요인을 유지한다.

> **wtime_length5는 요인이 아니라 독립 변수** <br>
> 교대근무 문항의 공통성이 0.9996, 적재량이 0.99976으로
> 제2요인을 단독으로 차지한다. 다른 문항과 공유하는 분산이 없다는 뜻이므로
> 지수로 구성할 대상이 아니라 단독 변수로 사용한다.

In [1]:
import numpy as np
import pandas as pd

# ═══════════════════════════════════════════════════════════
# 2.4 1차 변수 선택
#   원칙 : 제거 후보를 신호별로 수집하고, 보존 목록과의
#          교집합을 되살린다. 판단 근거를 표로 남긴다.
# ═══════════════════════════════════════════════════════════

# ── 보존 목록 A : 설계정보 (분석변수가 아니지만 반드시 유지) ──
DESIGN_KEEP = ['id', 'wt1', 'wt2', 'wt3', 'wt_std',
               'stratification', 'district', 'household']